# Chapter 4, Notebook 2: Fitting and Judging a Classifier

Notebook 1 ended with the logistic model, a curve that turns a customer's balance into a probability of default. This notebook answers two questions about it:

1. **How does the computer choose the curve?** (Section 1)
2. **Once we have it, how good is it?** (Sections 2–9)

We keep using ISL's `Default` data, with `balance` standardized as in Notebook 1. Only about 3% of customers default, and that rarity is exactly what makes judging a classifier interesting.

### By the end, you'll be able to

1. Explain fitting by maximum likelihood: choose the $\beta$'s that make the outcomes we observed as unsurprising as possible.
2. Explain why accuracy can mislead when one class is rare.
3. Build a confusion matrix and read accuracy, recall, precision, specificity, and the false positive rate from it.
4. Describe how moving the threshold trades one kind of mistake for the other.
5. Say what F1, the ROC curve, and AUC each summarize.
6. Choose a threshold from what each kind of mistake costs.

> **Positive means "defaulted."** Throughout this notebook the *positive* class is default (`Yes` → 1). "Flagging" a customer means predicting that they will default. "Positive" is not a judgment about whether the outcome is good.

## Setup

Run this cell first. It loads the data and fits the model **once**. Every demo below reuses these predicted probabilities; moving a slider only recounts and redraws, it never refits.

In [ ]:
from ch_4_demos_helpers import load_default, fit_balance_model

Default = load_default()
fit = fit_balance_model(Default)   # logistic regression of default01 on standardized balance

<details>
<summary>What's inside <code>fit_balance_model</code>?</summary>

It standardizes `balance` exactly as Notebook 1 did, then fits scikit-learn's `LogisticRegression(C=np.inf)`. By default scikit-learn adds a penalty that shrinks the coefficients (Chapter 6 covers this). Setting `C=np.inf` turns the penalty off, so we get the plain maximum likelihood fit that ISL and `statsmodels` report.

</details>

## 0 · Recap: the model from Notebook 1

The logistic model gives the probability of default as

$$p(X) = \frac{e^{\beta_0 + \beta_1 X}}{1 + e^{\beta_0 + \beta_1 X}}$$

where $X$ is standardized balance: how many standard deviations a customer's balance is above the average. Here is the fitted curve over the data. Each dot is one customer, nudged up or down a little so that 10,000 points don't collapse onto two lines.

In [ ]:
from ch_4_demos_helpers import plot_recap

plot_recap(fit);

We have the shape. **But how did the computer choose $\beta_0$ and $\beta_1$?**

## 1 · Fitting the model

### Demo 1.1: Which curve explains the data?

Two candidate curves for the same customers. One is the computer's fit, and the other is a guess.

**Ask:** Which curve better explains the data? What should "better" mean?

<details>
<summary>Answer</summary>

The better curve gives a **high** probability to the people who did default and a **low** probability to the people who didn't.

Curve A rises too late and too gently. At a balance 2.5 standard deviations above average it gives a 28% chance of default, but most customers there did default; curve B gives them 64%. Every defaulter curve A called unlikely is a surprise, and those surprises add up.

</details>

In [ ]:
from ch_4_demos_helpers import which_curve_demo

which_curve_demo(fit)

**Reading the score.** For each customer, look up the probability the curve gave to *what actually happened*: $p(x_i)$ if they defaulted, $1 - p(x_i)$ if they didn't. The **log-likelihood** adds up the logs of those probabilities. Every probability is at most 1, so every log is at most 0, and the total is negative. **Closer to 0 is better.** One customer the curve gave a 1% chance to what actually happened costs $\ln 0.01 \approx -4.6$ points; one it gave a 90% chance costs only $\ln 0.9 \approx -0.1$.

> **Maximum likelihood:** choose the $\beta$'s that make the outcomes we actually observed as probable, as unsurprising, as possible.

This is **not** least squares. We aren't measuring vertical distances from the dots to the curve. Appendix A3 shows why that would be the wrong yardstick.

### Demo 1.2: Fit it yourself (optional)

Move the sliders to find the curve with the best log-likelihood you can. Points grow when the curve finds them surprising, so try to shrink the big ones. When you've done your best, reveal the computer's answer.

In [ ]:
from ch_4_demos_helpers import hand_fit_demo

hand_fit_demo(fit)

However close you got, you can't beat the computer's score: it is the *maximum* likelihood. The computer doesn't search with sliders, though. Appendix A4 shows how it finds the top.

## 2 · Accuracy Misleads 

The model gives a probability, but the bank needs a **decision**: approve the customer, or flag them for review. For now we use the obvious rule:

$$\text{flag a customer when } p(X) \ge 0.5$$

The naive metric (or score) is **accuracy**: the fraction of customers we classify correctly. Before scoring the model, here is a classifier that doesn't even look at the data.

### Demo 2.1: The "always no" classifier

Two arrays from the setup cell do the work here: `fit.y` holds what actually happened (1 = defaulted, 0 = didn't), and `fit.p` holds the model's predicted probability of default for each customer. scikit-learn's `accuracy_score` compares actual outcomes to predicted ones and returns the fraction that match.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

always_no = np.zeros_like(fit.y)   # predict 0 ("won't default") for every customer

print(f'accuracy: {accuracy_score(fit.y, always_no):.2%}')
print(f'defaulters caught: {((always_no == 1) & (fit.y == 1)).sum()} of {fit.y.sum()}')

**Ask:** I have a classifier that's 96.7% accurate. Is it good?

<details>
<summary>Answer</summary>

It never catches a single defaulter, so it's useless to the bank. Only 333 of the 10,000 customers (3.3%) defaulted, so saying "no" to everyone is right 96.7% of the time.

</details>

Now the logistic model's accuracy: `fit.p >= 0.5` is `True` for every customer we flag, and `astype(int)` turns that into 1s and 0s, the same coding as `fit.y`.

In [ ]:
flagged = (fit.p >= 0.5).astype(int)   # predict 1 ("will default") when p(X) >= 0.5

print(f'accuracy: {accuracy_score(fit.y, flagged):.2%}')
print(f'defaulters caught: {((flagged == 1) & (fit.y == 1)).sum()} of {fit.y.sum()}')

The model is barely more accurate (97.25% vs 96.67%), but it catches 100 of the 333 defaulters instead of none. Accuracy can hardly tell the two classifiers apart.

When one class is rare, accuracy rewards ignoring it, and the rare class is usually the one we care about. To see what a classifier is really doing, we have to count the *kinds* of mistakes it makes.

*Good place to stop.*

## 3 · Two kinds of mistakes

Every customer lands in one of four groups:

| | We flagged them | We approved them |
|---|---|---|
| **Defaulted** | **True positive (TP):** a caught defaulter | **False negative (FN):** a missed defaulter. The bank loses the money. |
| **Didn't default** | **False positive (FP):** a false alarm. We bothered or turned away a good customer. | **True negative (TN):** correctly left alone |

The second word (*positive* or *negative*) is what we **predicted**. The first word (*true* or *false*) says whether that prediction was **right**.

### Demo 3.1: The outcome-colored scatter

The x-axis is the model's predicted probability; the y-axis is what actually happened. Everyone to the right of the dashed threshold line gets flagged. Each color is one of the four groups, and these four colors mean the same thing everywhere in this notebook.

In [ ]:
from ch_4_demos_helpers import plot_outcome_scatter

plot_outcome_scatter(fit, threshold=0.5);

**Ask:** Which mistake costs the bank more? Would a hospital screening for a disease answer the same way?

<details>
<summary>Answer</summary>

For the bank, a missed defaulter (FN) usually costs much more, the whole unpaid balance, than a false alarm (FP), which costs a review and some goodwill.

A hospital screening for a serious disease feels the same way, even more strongly: a missed case can be deadly, while a false alarm leads to a follow-up test. A spam filter is the opposite. Sending a real email to the spam folder (FP) is worse than letting one spam message through (FN).

The right trade-off depends on what each mistake costs. Section 9 comes back to this.

</details>

*Good place to stop.*

## 4 · The confusion matrix

A **confusion matrix** is just a tally of the scatter: count the points in each of the four regions.

### Demo 4.1: From scatter to table

Click **Count next cell** to tally one region at a time. Rows are what actually happened, and columns are what the model predicted. This is the layout scikit-learn uses.

> **Reading ISL:** ISL's tables (for example, Table 4.4) put the *predicted* class on the rows and the *actual* class on the columns, the other way around from here. Always check the labels before reading a confusion matrix.

In [ ]:
from ch_4_demos_helpers import confusion_build_demo

confusion_build_demo(fit)

**Ask:** Where would the "always no" classifier's results land in this table? (Then click the **Always "no"** toggle.)

<details>
<summary>Answer</summary>

Everything lands in the **not flagged** column: 9,667 true negatives and 333 false negatives. The flagged column is empty, because it never flags anyone.

</details>

scikit-learn builds the same table with `confusion_matrix`. Its rows are actual (0, 1) and its columns are predicted (0, 1):

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(fit.y, fit.p >= 0.5)

# Learning Check Practice

Here is the fitted logistic model from the recap again.

In [ ]:
plot_recap(fit);

Suppose the bank uses this rule:

- if $\hat p(\texttt{balance}) \ge 0.5$, we predict **default**
- if $\hat p(\texttt{balance}) < 0.5$, we predict **no default**

Recall that $Y = 1$ means the customer actually defaulted and $Y = 0$ means they didn't.

**Question 1.** An account has balance (standardized) $= -1$ and $Y = 0$. For this account, the logistic model plotted above predicts a:

- true positive
- true negative
- false positive
- false negative

<details>
<summary>Answer</summary>

**True negative.**

*What the words mean.* In this problem, **positive** means the model predicts **default** (the customer gets flagged), and **negative** means it predicts **no default**. "Positive" doesn't mean good news. It just means the model says the thing encoded as 1, a default, will happen. **True** means the prediction matched what actually happened (according to the dataset for this account/person/example), and **false** means it didn't. So the *positive/negative* is what the model *predicted*, and *true/false* says whether that prediction was *correct*.

*How to get the answer.*

1. **Find the prediction.** Find balance $= -1$ on the x-axis and go up to the curve. The curve's y-coordinate there is the predicted probability ($\hat p \approx 0.0002$), well below 0.5, so the model predicts no default: a **negative** prediction.
2. **Check it against $Y$.** $Y = 0$, so the customer didn't default. The prediction was right, so it's **true**.

True + negative = **true negative**.

</details>

**Question 2.** An account has balance (standardized) $= 1$ and $Y = 1$. For this account, the logistic model plotted above predicts a:

- true positive
- true negative
- false positive
- false negative

<details>
<summary>Answer</summary>

**False negative.**

1. **Find the prediction.** At balance $= 1$ the curve is still near the bottom ($\hat p \approx 0.03$). That's below 0.5, so the model predicts no default: a **negative**.
2. **Check it against $Y$.** $Y = 1$, so the customer actually defaulted. The prediction was wrong, so it's **false**.

This is a missed defaulter, the costly mistake for the bank.

</details>

**Question 3.** An account has balance (standardized) $= 3$ and $Y = 1$. For this account, the logistic model plotted above predicts a:

- true positive
- true negative
- false positive
- false negative

<details>
<summary>Answer</summary>

**True positive.**

1. **Find the prediction.** At balance $= 3$ the curve is high ($\hat p \approx 0.87$). That's at least 0.5, so the model predicts default: a **positive**.
2. **Check it against $Y$.** $Y = 1$, so the customer did default. The prediction was right, so it's **true**.

This is a caught defaulter.

</details>

**Question 4.** An account has balance (standardized) $= 2.5$ and $Y = 0$. For this account, the logistic model plotted above predicts a:

- true positive
- true negative
- false positive
- false negative

<details>
<summary>Answer</summary>

**False positive.**

1. **Find the prediction.** The curve crosses 0.5 at a balance of about 2.3. At balance $= 2.5$ it's a little above that ($\hat p \approx 0.64$), so the model predicts default: a **positive**.
2. **Check it against $Y$.** $Y = 0$, so the customer didn't default. The prediction was wrong, so it's **false**.

This is a false alarm: the bank flags a customer who would have paid.

</details>

*Good place to stop.*

# Section 2

## 5 · Metrics as questions

Each metric answers one plain question about the confusion matrix.

| Metric | Question it answers | Formula |
|---|---|---|
| Accuracy | What fraction of all our decisions were right? | (TP + TN) / total |
| Recall | Of the actual defaulters, what fraction did we catch? | TP / (TP + FN) |
| Precision | Of the customers we flagged, what fraction actually defaulted? | TP / (TP + FP) |
| Specificity | Of the non-defaulters, what fraction did we leave alone? | TN / (TN + FP) |
| False positive rate | Of the non-defaulters, what fraction did we wrongly flag? | FP / (TN + FP) = 1 − specificity |

### Demo 5.1: Metric highlighter

Pick a metric. The cells on **top** of its fraction (the numerator) light up, the cells on the **bottom** (the denominator) are outlined, and the cells it ignores fade out.

In [ ]:
from ch_4_demos_helpers import metric_highlighter

metric_highlighter(fit)

**Things to notice:**

- Recall and precision both have TP on top. Recall divides by a **row** (everyone who actually defaulted). Precision divides by a **column** (everyone we flagged).
- Switch to **Always "no"**: recall is 0, and precision is 0/0, which is undefined because nobody was flagged. (scikit-learn warns and reports 0 in this case.)
- Accuracy is the only metric here that uses all four cells, which is how 9,667 true negatives can drown out everything else.

**Other names you'll see.** Different fields named these metrics independently, so most have several names. This notebook only uses the names in the first column.

| Name used here | Also called |
|---|---|
| Recall | sensitivity, true positive rate (TPR), power, hit rate |
| Precision | positive predictive value (PPV) |
| Specificity | true negative rate (TNR) |
| False positive rate | 1 − specificity, fall-out, Type I error rate |
| 1 − recall | false negative rate, miss rate, Type II error rate |

*Good place to stop.*

## 6 · The threshold

The logistic model gives a probability. Turning it into a decision takes a **threshold**, and nothing forces that threshold to be 0.5.

### Demo 6.1: The threshold slider

Drag the threshold, or use the preset buttons. The scatter, the confusion matrix, and the metrics all update together. The ▲ and ▼ arrows show which way each metric moved since the last change. The model and its probabilities never change; only the cutoff moves.

In [ ]:
from ch_4_demos_helpers import threshold_demo

threshold_demo(fit)

**Ask:** Move the threshold from 0.5 to 0.2. **Predict first:** what happens to recall? To precision? To accuracy?

<details>
<summary>Answer</summary>

- **Recall rises**, from 0.30 to 0.60: we catch 199 defaulters instead of 100.
- **Precision falls**, from 0.70 to 0.43: we raise 263 false alarms instead of 42.
- **Accuracy falls slightly**, from 97.3% to 96.0%, even though we now catch twice as many defaulters. The 221 new false alarms outweigh the 99 new catches, and accuracy counts every mistake the same.

</details>

**Ask:** Can any threshold make both recall and precision perfect?

<details>
<summary>Answer</summary>

No. Along the x-axis the colors overlap: some defaulters got lower predicted probabilities than some non-defaulters. Wherever the line goes, it either misses some defaulters or flags some non-defaulters. Perfect recall needs a threshold below *every* defaulter's probability, and that flags thousands of good customers.

</details>

**Ask:** Which threshold would you choose?

<details>
<summary>Answer</summary>

We can't say yet: it depends on what each kind of mistake costs the bank. Lower thresholds trade false alarms for caught defaulters, and only the costs say whether that trade is worth it. Section 9 answers this.

</details>

**The big idea:** 0.5 is a choice, not a law. Moving the threshold trades one kind of mistake for the other.

*Good place to stop.*

## 7 · F1

We often want a single number that rewards a classifier for doing well on **both** recall and precision. The usual choice is **F1**:

$$F_1 = \frac{2 \cdot \text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$$

### Demo 7.1: Why not just average?

In [ ]:
from ch_4_demos_helpers import f1_vs_average_demo

f1_vs_average_demo()

**Ask:** Click the extreme preset (precision 1.0, recall 0.02). Which number better describes a classifier that catches 2% of defaulters?

<details>
<summary>Answer</summary>

F1. The plain average (0.51) makes this classifier sound half good. F1 (about 0.04) says it's nearly useless. F1 stays low unless **both** precision and recall are decent, so a classifier can't hide being terrible at one behind being perfect at the other.

</details>

### Demo 7.2: F1 at every threshold

The threshold slider again, now with F1 in the readout and a plot of F1 at every threshold.

In [ ]:
threshold_demo(fit, show_f1=True)

For this model, F1 peaks around a threshold of 0.32, not 0.5. Still, F1 is one particular compromise: it counts a missed defaulter and a false alarm as equally bad, and it ignores true negatives. It doesn't know what the bank's mistakes actually cost, either.

## 8 · ROC curves and AUC

F1 judges one threshold at a time. An **ROC curve** records the whole threshold slider at once. Every threshold gives one point:

- x = **false positive rate** (the share of non-defaulters we flag)
- y = **recall** (the share of defaulters we catch)

### Demo 8.1: ROC tracer

Move the slider and watch the point. **Sweep** moves the threshold from high to low and traces the curve. **Show full curve** draws every threshold at once.

In [ ]:
from ch_4_demos_helpers import roc_tracer

roc_tracer(fit)

A very high threshold flags nobody, at the bottom-left corner $(0, 0)$. A very low threshold flags everybody, at the top-right corner $(1, 1)$. A good model climbs quickly: it catches most defaulters before it flags many non-defaulters. Random guessing moves along the diagonal, catching defaulters only as fast as it flags everybody else.

**AUC**, the *area under the curve*, sums up the whole curve in one number. It has a plain meaning:

> **AUC** = the chance that a randomly chosen defaulter got a higher predicted probability than a randomly chosen non-defaulter.

An AUC of 0.5 is a coin flip, and 1.0 is perfect. Our model's AUC is about 0.95. AUC doesn't depend on any threshold. It only asks whether the model *ranks* defaulters above non-defaulters.

### Demo 8.2: Comparing models

Toggle the curves on and off:

- logistic regression on **balance** (our model),
- logistic regression on **income** only,
- **random** scores.

In [ ]:
from ch_4_demos_helpers import roc_comparison

roc_comparison(Default, fit)

**Ask:** Why does the income model's curve sit so close to the diagonal?

<details>
<summary>Answer</summary>

Defaulters and non-defaulters have very similar incomes (look back at the boxplots in Notebook 1), so ranking customers by income is barely better than ranking them at random. Its AUC is about 0.53. The random scores land near 0.5 (about 0.48 here, below 0.5 just by chance).

</details>

## 9 · Choosing a threshold with costs

To choose a threshold, put a dollar cost on each kind of mistake and add them up:

$$\text{total cost} = (\text{cost of a missed defaulter}) \times \text{FN} + (\text{cost of a false alarm}) \times \text{FP}$$

The starting values are made up, but they're reasonable:

- **\$2,000 per missed defaulter.** The average defaulter in this data owes about \$1,750, and collecting costs money too.
- **\$200 per false alarm.** That covers a manual review, plus the chance an annoyed good customer takes their business elsewhere.

### Demo 9.1: Cost sliders

The top plot shows the total cost at every threshold, with the cheapest one marked. The scatter and confusion matrix below use that cheapest threshold.

In [ ]:
from ch_4_demos_helpers import cost_demo

cost_demo(fit)

**Ask:** If false negatives get more expensive, which way does the best threshold move? Why?

<details>
<summary>Answer</summary>

**Down.** When each missed defaulter costs more, it's worth putting up with more false alarms to catch more defaulters, so we flag more people.

Here's a rule of thumb that needs no calculus. Flag a customer when flagging costs less, on average, than approving:

$$\underbrace{p \times (\text{cost of FN})}_{\text{expected cost of approving}} > \underbrace{(1 - p) \times (\text{cost of FP})}_{\text{expected cost of flagging}} \quad\Longleftrightarrow\quad p > \frac{\text{cost of FP}}{\text{cost of FP} + \text{cost of FN}}$$

With the bank costs that cutoff is $200 / 2{,}200 \approx 0.09$, close to the 0.08 the demo finds. With equal costs it's 0.5, close to the demo's 0.52. The rule works only as well as the probabilities are accurate, so the demo's answers differ slightly.

</details>

### Wrapping up

- **The best threshold depends on the costs, not on the model alone.** The same model can call for very different thresholds at a bank, a hospital, and an email provider.
- **A caveat:** we measured everything on the same customers the model was fit to, as ISL does in Chapter 4. Real evaluation uses customers the model never saw during fitting. Chapter 5 covers how to do that properly.
- **In Friday's lab** you'll pick a threshold for a classifier and justify it.

---

# Appendix: Deeper maximum likelihood (optional)

This appendix is for the curious and for the Friday lab. Nothing in Sections 0–9 depends on it.

## A1 · The coin

You flip a coin 10 times and get 7 heads. What's your best guess for $p$, the chance of heads? For any candidate $p$, the probability of seeing exactly this sequence of flips is

$$L(p) = p^7 (1 - p)^3$$

This is the **likelihood** of $p$. Here it is for three candidates:

In [ ]:
from ch_4_demos_helpers import coin_table, coin_demo

coin_table()

$p = 0.7$ makes what we saw most probable. Slide $p$ to explore the whole curve, then change the number of heads and watch the peak move to heads / 10.

In [ ]:
coin_demo()

**The key flip:** *probability* fixes $p$ and asks about the data ("if the coin is fair, how likely are 7 heads?"). *Likelihood* fixes the data we already saw and asks about $p$ ("given 7 heads, how well does each $p$ explain it?").

## A2 · Every customer is a coin

Logistic regression treats each customer as a coin with their own chance of "heads" (default), $p(x_i)$, which depends on their balance. The likelihood multiplies the probability of what happened to every customer (ISL equation 4.5):

$$\ell(\beta_0, \beta_1) = \prod_{i:\,y_i = 1} p(x_i) \prod_{i':\,y_{i'} = 0} \bigl(1 - p(x_{i'})\bigr)$$

The first product runs over the customers who defaulted and the second over those who didn't. Demo 1.2 (hand-fitting) is this formula with sliders: each slider position is a choice of $\beta_0, \beta_1$, and the score is the log of $\ell$.

## A3 · Why take the log, and why not least squares?

**Why the log.** Multiply 10,000 probabilities, each below 1, and the result is too small for a computer to store. Taking logs turns the product into a sum, which stays a sensible size:

In [ ]:
from ch_4_demos_helpers import underflow_table, plot_penalties

underflow_table(fit)

With all 10,000 customers the product **underflows** to exactly 0.0, and every curve would get the same useless score. The sum of logs, about −798, is the log-likelihood from Section 1. Because the log only ever goes up when its input goes up, the $\beta$'s that maximize the log-likelihood also maximize the likelihood.

**Why not least squares.** Take one customer who defaulted, and suppose the model gave them a predicted probability $p$. Here are the two penalties:

In [ ]:
plot_penalties();

Squared error tops out at 1, so calling a defaulter 0.1% likely to default costs barely more than calling them 30% likely. Log loss grows without limit as $p \to 0$. **Confident mistakes should be catastrophic,** and maximum likelihood makes them so.

## A4 · The loss surface and gradient descent

Every pair $(\beta_0, \beta_1)$ has a score. Flip the sign of the log-likelihood (so lower is better), average it over the customers, and draw it as a contour map: a valley with the best fit at the bottom (the star).

**Gradient descent** starts somewhere and repeatedly takes a small step downhill. Choose a starting point and a step size, then click **Step** or **Run**.

**Things to try:**

1. The default step size (10) walks steadily toward the star. Try 40: the steps overshoot and the path bounces across the valley.
2. Switch to **raw dollars**. Now $\beta_1$ is "per dollar," so a tiny change in $\beta_1$ changes the curve enormously, while $\beta_0$ barely matters. Any step small enough not to blow up in the $\beta_1$ direction is far too small to make progress toward the right $\beta_0$, and the path stalls. Standardizing puts both coefficients on a similar footing, which is one big reason we standardized.

In [ ]:
from ch_4_demos_helpers import gradient_descent_demo

gradient_descent_demo(fit)

Real software (including scikit-learn and `statsmodels`) uses smarter methods that also use how the surface *curves*, so it reaches the bottom in a handful of steps. The picture is the same, though: a valley with one lowest point, found by going downhill.

## A5 · Connections

- **Least squares is maximum likelihood, too.** If a regression's errors are normally distributed, the line that maximizes the likelihood is exactly the least squares line. Chapter 3's method was maximum likelihood all along, for a different kind of noise.
- **Residuals return.** The downhill direction gradient descent follows is built from $\sum_i (y_i - p_i)\,x_i$: each customer's *residual* (outcome minus predicted probability), weighted by their balance.
- **Neural networks.** The average negative log-likelihood is the **cross-entropy loss** used to train neural network classifiers later in the course. It's the same score, applied to a more flexible model.